<a href="https://colab.research.google.com/github/AzaharaAED/Proyecto_ecosistema/blob/main/bascula.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# 1. CREAR TABLA DE USUARIOS

def crear_tabla_usuarios():
    usuarios_df = pd.DataFrame([
        ["u01", "Valero", 36, "M", 177],
        ["u02", "1hijo", 7, "M", 130],
        ["u03", "2hijo", 5, "M", 108],
        ["u04", "3hijo", 3, "F", 96],
    ], columns=["user_id", "nombre", "edad", "sexo", "altura_cm"])

    return usuarios_df


# 2. CREAR HISTÓRICO DE BÁSCULA

# 1. CREAR TABLA DE USUARIOS

def crear_tabla_usuarios():
    usuarios_df = pd.DataFrame([
        ["u01", "Valero", 36, "M", 177],
        ["u02", "1hijo", 7, "M", 130],
        ["u03", "2hijo", 5, "M", 108],
        ["u04", "3hijo", 3, "F", 96],
    ], columns=["user_id", "nombre", "edad", "sexo", "altura_cm"])

    return usuarios_df


# 2. CREAR HISTÓRICO DE BÁSCULA

def crear_bascula_df():
    bascula_df = pd.DataFrame([
        ["u01", "2026-03-14 08:30", 76.5, 24.4, 19.5, 34.2],
        ["u01", "2026-03-15 08:32", 76.4, 24.4, 19.4, 34.3],
        ["u01", "2026-03-16 08:35", 76.3, 24.3, 19.3, 34.4],

        ["u02", "2026-03-14 09:00", 27.8, 16.4, 18.2, 10.8],
        ["u02", "2026-03-15 09:05", 27.9, 16.5, 18.1, 10.9],
        ["u02", "2026-03-16 09:02", 28.0, 16.6, 18.0, 11.0],

        ["u03", "2026-03-14 08:10", 18.5, 15.9, 19.0, 8.2],
        ["u03", "2026-03-15 08:12", 18.6, 16.0, 18.9, 8.3],
        ["u03", "2026-03-16 08:14", 18.7, 16.0, 18.8, 8.4],

        ["u04", "2026-03-14 07:50", 14.2, 15.4, 20.5, 6.4],
        ["u04", "2026-03-15 07:55", 14.3, 15.5, 20.4, 6.5],
        ["u04", "2026-03-16 07:52", 14.4, 15.6, 20.3, 6.6],
    ], columns=["user_id", "timestamp", "weight", "bmi", "body_fat", "muscle_mass"])

    bascula_df["timestamp"] = pd.to_datetime(bascula_df["timestamp"])
    return bascula_df


# 3. FUNCIONES DE BÚSQUEDA DE USUARIO

def obtener_user_id_por_nombre(nombre, usuarios_df):
    nombre = str(nombre).strip().lower()

    coincidencias = usuarios_df[
        usuarios_df["nombre"].astype(str).str.strip().str.lower() == nombre
    ]

    if coincidencias.empty:
        return None

    return coincidencias.iloc[0]["user_id"]


def obtener_user_id_por_indice(indice, usuarios_df):
    try:
        indice = int(indice)
    except (TypeError, ValueError):
        return None

    if indice < 1 or indice > len(usuarios_df):
        return None

    return usuarios_df.iloc[indice - 1]["user_id"]


def obtener_info_usuario(user_id, usuarios_df):
    coincidencias = usuarios_df[usuarios_df["user_id"] == user_id]

    if coincidencias.empty:
        return None

    fila = coincidencias.iloc[0]

    return {
        "user_id": fila["user_id"],
        "nombre": fila["nombre"],
        "edad": int(fila["edad"]),
        "sexo": fila["sexo"],
        "altura_cm": int(fila["altura_cm"])
    }


# 4. FUNCIONES DE BÁSCULA

def filtrar_bascula_por_usuario(user_id, bascula_df):
    usuario_df = bascula_df[bascula_df["user_id"] == user_id].copy()

    if usuario_df.empty:
        return pd.DataFrame(columns=bascula_df.columns)

    return usuario_df.sort_values("timestamp").reset_index(drop=True)


def obtener_ultimo_registro_bascula(user_id, bascula_df):
    usuario_df = filtrar_bascula_por_usuario(user_id, bascula_df)

    if usuario_df.empty:
        return None

    return usuario_df.iloc[-1]


def obtener_historico_bascula(user_id, bascula_df):
    return filtrar_bascula_por_usuario(user_id, bascula_df)


# 5. FORMATO LIMPIO PARA EL MODELO

def obtener_ultima_medicion_para_modelo(user_id, bascula_df):
    ultimo = obtener_ultimo_registro_bascula(user_id, bascula_df)

    if ultimo is None:
        return {
            "peso": None,
            "imc": None,
            "grasa": None,
            "musculo": None,
            "timestamp": None
        }

    return {
        "peso": float(ultimo["weight"]),
        "imc": float(ultimo["bmi"]),
        "grasa": float(ultimo["body_fat"]),
        "musculo": float(ultimo["muscle_mass"]),
        "timestamp": str(ultimo["timestamp"])
    }


def obtener_historico_para_modelo(user_id, bascula_df):
    historico = obtener_historico_bascula(user_id, bascula_df)

    if historico.empty:
        return []

    salida = []

    for _, fila in historico.iterrows():
        salida.append({
            "timestamp": str(fila["timestamp"]),
            "peso": float(fila["weight"]),
            "imc": float(fila["bmi"]),
            "grasa": float(fila["body_fat"]),
            "musculo": float(fila["muscle_mass"])
        })

    return salida


def obtener_datos_bascula_para_modelo(user_id, bascula_df):
    return {
        "user_id": user_id,
        "ultima_medicion": obtener_ultima_medicion_para_modelo(user_id, bascula_df),
        "historico": obtener_historico_para_modelo(user_id, bascula_df)
    }


# 6. FUNCIÓN AUXILIAR COMPLETA

def inicializar_modulo_bascula():
    usuarios_df = crear_tabla_usuarios()
    bascula_df = crear_bascula_df()

    return {
        "usuarios_df": usuarios_df,
        "bascula_df": bascula_df
    }